# NBS显著边均值与阅读行为的相关分析

## 分析说明
- 只对比 COM 和 TD 两组的 NBS 显著边
- 使用 NBS 显著边的均值作为特征
- 与所有阅读行为变量进行简单相关分析
- 包括 Pearson 和 Spearman 相关

In [1]:
# 导入必要的库
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 可视化
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'STHeiti']
plt.rcParams['axes.unicode_minus'] = False

print('库导入完成')

库导入完成


In [2]:
# =============================================================================
# 1. 配置路径
# =============================================================================
BASE_PATH = Path('/Users/gaozhenning/Desktop/CodeData/REST_COM-main')

CONN_PATHS = {
    'com': BASE_PATH / 'analysis/connectivity_analysis/com/results',
    'td': BASE_PATH / 'analysis/connectivity_analysis/td/results',
}

NBS_EDGES_PATH = BASE_PATH / 'reports/comparison/nbs/nbs_significant_edges.csv'
BEHAVIOR_PATH = BASE_PATH / '行为数据spss.csv'

OUTPUT_DIR = BASE_PATH / 'reports' / 'comparison' / 'correlation_nbs_mean'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'数据路径配置完成')
print(f'NBS边文件: {NBS_EDGES_PATH}')
print(f'行为数据: {BEHAVIOR_PATH}')
print(f'输出目录: {OUTPUT_DIR}')

数据路径配置完成
NBS边文件: /Users/gaozhenning/Desktop/CodeData/REST_COM-main/reports/comparison/nbs/nbs_significant_edges.csv
行为数据: /Users/gaozhenning/Desktop/CodeData/REST_COM-main/行为数据spss.csv
输出目录: /Users/gaozhenning/Desktop/CodeData/REST_COM-main/reports/comparison/correlation_nbs_mean


In [3]:
# =============================================================================
# 2. 加载NBS显著边
# =============================================================================
nbs_edges_df = pd.read_csv(NBS_EDGES_PATH)

# 筛选 COM vs TD 的显著边
com_td_edges = nbs_edges_df[
    (nbs_edges_df['pair'] == 'com_vs_td') & 
    (nbs_edges_df['freq_band'] == 'gamma')
]

print(f'NBS显著边总数: {len(nbs_edges_df)}')
print(f'COM vs TD gamma频段显著边数: {len(com_td_edges)}')

if len(com_td_edges) > 0:
    print('\n显著边列表:')
    print(com_td_edges[['ch1', 'ch2', 't_value']].to_string())

NBS显著边总数: 97
COM vs TD gamma频段显著边数: 55

显著边列表:
    ch1  ch2   t_value
1   CP5  PO4  4.579617
5    C4   F1  4.126312
6   CP5   P2  4.094059
8   TP7  CP4  3.962353
11  CP5  PO8  3.888181
13  TP7  TP8  3.747248
14  CP6  CP3  3.736571
15  CP1  CP4  3.688314
17   P4  CP3  3.643706
19  CP5   O2  3.580841
22  TP7  FCz  3.565950
23   P2  AF4  3.563107
24   F8  FT7  3.559946
25   P1  FC4  3.559206
27  FC6  CP4  3.543822
28   T8   P5  3.526914
30  CP5   F8  3.511762
34   T8  TP7  3.457374
35   P5  TP8  3.455180
39  PO4  TP8  3.405258
40   P7   T8  3.403286
41   P7  CP4  3.373976
43  FC1  FC6  3.351179
45  CP6   F8  3.349129
46  POz   F6  3.332197
47  POz  AF4  3.320537
48  CP6  TP7  3.318480
50   P4  Fp2  3.312804
52   F8   F4  3.287035
53  Fp1  AF3  3.284502
54   P7   C4  3.247596
55   F8   C5  3.243394
56   T8   F1  3.233343
57   C3   P8  3.228958
58  FC6   P1  3.228106
61   C1  TP7  3.220004
65  FC3  CP4  3.199409
66  FC5  CP4  3.193835
69   F1   C6  3.171561
70  CP6  FC3  3.170879
72  FT7   

In [4]:
# =============================================================================
# 3. 加载行为数据
# =============================================================================
behav_df = pd.read_csv(BEHAVIOR_PATH)
behav_df = behav_df.dropna(subset=['被试编号', '分组'])

# 标准化分组名称为小写
behav_df['group_lower'] = behav_df['分组'].str.lower()

# 创建从被试ID到分组的映射
subject_to_group = {}
for _, row in behav_df.iterrows():
    subj_id = str(row['被试编号'])[3:]  # 去掉BZK
    subject_to_group[subj_id] = row['group_lower']

print(f'行为数据总样本数: {len(behav_df)}')
print(f'分组分布:')
print(behav_df['group_lower'].value_counts())

行为数据总样本数: 34
分组分布:
group_lower
com     14
adhd    12
td       8
Name: count, dtype: int64


In [5]:
# =============================================================================
# 4. 定义行为变量
# =============================================================================
BEHAVIOR_VARS = [
    '150字',
    '1分钟阅读平均',
    '数字RAN均值',
    '物体RAN均值',
    '阅读流畅性',
    '音位删除',
    '部首意识',
]

# 清洗行为数据中的特殊值
for col in BEHAVIOR_VARS:
    if col in behav_df.columns:
        # 将带括号的数值提取出来（如 "75（71.5）" -> 75）
        behav_df[col] = behav_df[col].astype(str).str.replace(r'[^0-9.\-]', '', regex=True)
        behav_df[col] = pd.to_numeric(behav_df[col], errors='coerce')
        # 处理-999为缺失值
        behav_df.loc[behav_df[col] == -999, col] = np.nan

print('行为变量列表:')
for var in BEHAVIOR_VARS:
    if var in behav_df.columns:
        n_valid = behav_df[var].notna().sum()
        print(f'  {var}: {n_valid} 个有效样本')

行为变量列表:
  150字: 34 个有效样本
  1分钟阅读平均: 34 个有效样本
  数字RAN均值: 34 个有效样本
  物体RAN均值: 34 个有效样本
  阅读流畅性: 34 个有效样本
  音位删除: 33 个有效样本
  部首意识: 33 个有效样本


In [6]:
# =============================================================================
# 5. 提取NBS显著边均值
# =============================================================================
groups = ['com', 'td']

# 获取通道名和索引映射
npz_sample = np.load(CONN_PATHS[groups[0]] / 'connectivity_results.npz', allow_pickle=True)
channel_names = list(npz_sample['channel_names'])
ch2idx = {ch: i for i, ch in enumerate(channel_names)}

# 收集所有被试的NBS边均值
all_nbs_means = []
all_subj_ids = []
all_groups = []

for group in groups:
    npz = np.load(CONN_PATHS[group] / 'connectivity_results.npz', allow_pickle=True)
    matrices = npz['gamma']  # 使用gamma频段
    
    for subj_idx, subj_id in enumerate(npz['subject_ids']):
        subj_id_str = str(subj_id)
        mat = matrices[subj_idx]
        
        # 提取NBS显著边的值并计算均值
        edge_values = []
        for _, row in com_td_edges.iterrows():
            i = ch2idx.get(row['ch1'])
            j = ch2idx.get(row['ch2'])
            if i is not None and j is not None:
                edge_values.append(float(mat[i, j]))
        
        nbs_mean = np.mean(edge_values) if edge_values else 0.0
        
        all_nbs_means.append(nbs_mean)
        all_subj_ids.append(subj_id_str)
        all_groups.append(group)

nbs_mean_df = pd.DataFrame({
    'subject_id': all_subj_ids,
    'group': all_groups,
    'nbs_mean': all_nbs_means
})

# 匹配行为数据
nbs_mean_df['matched_group'] = nbs_mean_df['subject_id'].map(subject_to_group)
nbs_mean_df = nbs_mean_df[nbs_mean_df['matched_group'].isin(groups)]

# 与行为数据合并
merged_df = nbs_mean_df.copy()
for _, row in behav_df.iterrows():
    subj_id_behav = str(row['被试编号'])[3:]
    idx = merged_df[merged_df['subject_id'] == subj_id_behav].index
    if len(idx) > 0:
        for var in BEHAVIOR_VARS:
            merged_df.loc[idx, var] = row[var]

print(f'NBS均值样本数: {len(merged_df)}')
print(f'组别分布:')
print(merged_df['group'].value_counts())
print(f'\n数据概览:')
print(merged_df[['subject_id', 'group', 'nbs_mean']].head(10))

NBS均值样本数: 22
组别分布:
group
com    14
td      8
Name: count, dtype: int64

数据概览:
  subject_id group  nbs_mean
0        001   com  0.282097
1        020   com  0.313629
2        021   com  0.259352
3        022   com  0.299946
4        024   com  0.306263
5        047   com  0.373689
6        049   com  0.324134
7        051   com  0.244000
8        072   com  0.330168
9        073   com  0.251640


In [7]:
# =============================================================================
# 6. 简单相关分析
# =============================================================================
def correlation_analysis(x, y, method='pearson'):
    """计算两个变量之间的相关性"""
    valid_mask = ~np.isnan(x) & ~np.isnan(y)
    x_valid = x[valid_mask]
    y_valid = y[valid_mask]
    
    if len(x_valid) < 5:
        return np.nan, np.nan, 0
    
    if method == 'pearson':
        r, p = stats.pearsonr(x_valid, y_valid)
    else:
        r, p = stats.spearmanr(x_valid, y_valid)
    
    return r, p, len(x_valid)

def get_significance_marker(p):
    """获取显著性标记"""
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    elif p < 0.1:
        return '†'
    else:
        return ''

# 计算Pearson和Spearman相关
pearson_results = []
spearman_results = []

for var in BEHAVIOR_VARS:
    if var not in merged_df.columns:
        continue
    
    x = merged_df['nbs_mean'].values.astype(float)
    y = merged_df[var].values.astype(float)
    
    # Pearson相关
    r_pearson, p_pearson, n_pearson = correlation_analysis(x, y, 'pearson')
    pearson_results.append({
        'behavior_var': var,
        'n': n_pearson,
        'r': r_pearson,
        'p_value': p_pearson,
        'significance': get_significance_marker(p_pearson)
    })
    
    # Spearman相关
    r_spearman, p_spearman, n_spearman = correlation_analysis(x, y, 'spearman')
    spearman_results.append({
        'behavior_var': var,
        'n': n_spearman,
        'r': r_spearman,
        'p_value': p_spearman,
        'significance': get_significance_marker(p_spearman)
    })

pearson_df = pd.DataFrame(pearson_results)
spearman_df = pd.DataFrame(spearman_results)

print('=' * 70)
print('NBS显著边均值与阅读行为的简单相关分析结果')
print('=' * 70)
print(f'\n样本数: COM={len(merged_df[merged_df["group"]=="com"])}, TD={len(merged_df[merged_df["group"]=="td"])}')
print(f'NBS显著边数: {len(com_td_edges)}')

NBS显著边均值与阅读行为的简单相关分析结果

样本数: COM=14, TD=8
NBS显著边数: 55


In [8]:
# =============================================================================
# 7. 显示Pearson相关结果
# =============================================================================
print('\n' + '=' * 70)
print('Pearson 相关分析结果')
print('=' * 70)
print(f'\n{"行为变量":<15} {"n":>6} {"r":>8} {"p值":>10} {"显著性":>8}')
print('-' * 50)

for _, row in pearson_df.sort_values('p_value').iterrows():
    print(f'{row["behavior_var"]:<15} {row["n"]:>6} {row["r"]:>8.4f} {row["p_value"]:>10.4f} {row["significance"]:>8}')

# 标记显著结果
sig_pearson = pearson_df[pearson_df['p_value'] < 0.05]
print(f'\n显著相关数量 (p<0.05): {len(sig_pearson)}')


Pearson 相关分析结果

行为变量                 n        r         p值      显著性
--------------------------------------------------
物体RAN均值             22   0.4274     0.0473        *
数字RAN均值             22   0.4224     0.0502        †
阅读流畅性               22  -0.3816     0.0797        †
1分钟阅读平均             22  -0.3308     0.1327         
150字                22  -0.2448     0.2722         
音位删除                21  -0.1957     0.3952         
部首意识                21   0.0695     0.7647         

显著相关数量 (p<0.05): 1


In [ ]:
# =============================================================================
# 9.5 可视化 - 带p值的相关性热图
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 准备热图数据 - Pearson
pearson_heatmap_data = pearson_df[['behavior_var', 'r', 'p_value']].copy()
pearson_heatmap_data = pearson_heatmap_data.sort_values('p_value')
r_values = pearson_heatmap_data['r'].values.reshape(-1, 1)
p_values = pearson_heatmap_data['p_value'].values.reshape(-1, 1)

# Pearson 热图
ax1 = axes[0]
im1 = ax1.imshow(r_values, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax1.set_xticks([0])
ax1.set_xticklabels(['r'])
ax1.set_yticks(range(len(pearson_heatmap_data)))
ax1.set_yticklabels(pearson_heatmap_data['behavior_var'].values)

# 添加r值和p值标注
for i in range(len(pearson_heatmap_data)):
    r = r_values[i, 0]
    p = p_values[i, 0]
    sig = get_significance_marker(p)
    
    # r值颜色
    text_color = 'white' if abs(r) > 0.5 else 'black'
    ax1.text(0, i, f'{r:.3f}{sig}', ha='center', va='center', color=text_color, fontsize=11, fontweight='bold')
    
    # p值标注在右侧
    p_color = 'red' if p < 0.05 else 'gray'
    ax1.text(0.8, i, f'p={p:.4f}', ha='left', va='center', color=p_color, fontsize=9)

ax1.set_title('Pearson Correlation: NBS Mean vs Behaviors\n(* p<0.05, ** p<0.01, *** p<0.001, † p<0.1)', fontsize=11)
cbar1 = plt.colorbar(im1, ax=ax1, shrink=0.8)
cbar1.set_label('r')

# 准备热图数据 - Spearman
spearman_heatmap_data = spearman_df[['behavior_var', 'r', 'p_value']].copy()
spearman_heatmap_data = spearman_heatmap_data.sort_values('p_value')
r_values_s = spearman_heatmap_data['r'].values.reshape(-1, 1)
p_values_s = spearman_heatmap_data['p_value'].values.reshape(-1, 1)

# Spearman 热图
ax2 = axes[1]
im2 = ax2.imshow(r_values_s, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax2.set_xticks([0])
ax2.set_xticklabels(['rho'])
ax2.set_yticks(range(len(spearman_heatmap_data)))
ax2.set_yticklabels(spearman_heatmap_data['behavior_var'].values)

# 添加rho值和p值标注
for i in range(len(spearman_heatmap_data)):
    r = r_values_s[i, 0]
    p = p_values_s[i, 0]
    sig = get_significance_marker(p)
    
    # r值颜色
    text_color = 'white' if abs(r) > 0.5 else 'black'
    ax2.text(0, i, f'{r:.3f}{sig}', ha='center', va='center', color=text_color, fontsize=11, fontweight='bold')
    
    # p值标注在右侧
    p_color = 'red' if p < 0.05 else 'gray'
    ax2.text(0.8, i, f'p={p:.4f}', ha='left', va='center', color=p_color, fontsize=9)

ax2.set_title('Spearman Correlation: NBS Mean vs Behaviors\n(* p<0.05, ** p<0.01, *** p<0.001, † p<0.1)', fontsize=11)
cbar2 = plt.colorbar(im2, ax=ax2, shrink=0.8)
cbar2.set_label('rho')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'correlation_heatmap_with_pvalues.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n带p值的热图已保存: {OUTPUT_DIR / "correlation_heatmap_with_pvalues.png"}')

In [9]:
# =============================================================================
# 8. 显示Spearman相关结果
# =============================================================================
print('\n' + '=' * 70)
print('Spearman 相关分析结果')
print('=' * 70)
print(f'\n{"行为变量":<15} {"n":>6} {"rho":>8} {"p值":>10} {"显著性":>8}')
print('-' * 50)

for _, row in spearman_df.sort_values('p_value').iterrows():
    print(f'{row["behavior_var"]:<15} {row["n"]:>6} {row["r"]:>8.4f} {row["p_value"]:>10.4f} {row["significance"]:>8}')

# 标记显著结果
sig_spearman = spearman_df[spearman_df['p_value'] < 0.05]
print(f'\n显著相关数量 (p<0.05): {len(sig_spearman)}')


Spearman 相关分析结果

行为变量                 n      rho         p值      显著性
--------------------------------------------------
数字RAN均值             22   0.5302     0.0111        *
物体RAN均值             22   0.4778     0.0245        *
1分钟阅读平均             22  -0.3909     0.0721        †
阅读流畅性               22  -0.3653     0.0946        †
音位删除                21  -0.3449     0.1257         
150字                22  -0.1825     0.4163         
部首意识                21  -0.0189     0.9352         

显著相关数量 (p<0.05): 2


In [15]:
# =============================================================================
# 9. 可视化 - 相关热图
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pearson 相关热图
ax1 = axes[0]
pearson_sorted = pearson_df.sort_values('p_value')
colors = ['red' if r < 0 else 'blue' for r in pearson_sorted['r']]
bars = ax1.barh(pearson_sorted['behavior_var'], pearson_sorted['r'], color=colors, alpha=0.7)
ax1.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax1.set_xlabel('Correlation (r)')
ax1.set_title('Pearson Correlation: NBS Mean vs Reading Behaviors')
ax1.set_xlim(-1, 1)

# 添加显著性标记
for i, (_, row) in enumerate(pearson_sorted.iterrows()):
    if row['significance']:
        ax1.text(row['r'] + 0.05 if row['r'] > 0 else row['r'] - 0.05, i,
                row['significance'], va='center', ha='left' if row['r'] > 0 else 'right')

# Spearman 相关热图
ax2 = axes[1]
spearman_sorted = spearman_df.sort_values('p_value')
colors = ['red' if r < 0 else 'blue' for r in spearman_sorted['r']]
bars = ax2.barh(spearman_sorted['behavior_var'], spearman_sorted['r'], color=colors, alpha=0.7)
ax2.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax2.set_xlabel('Correlation (rho)')
ax2.set_title('Spearman Correlation: NBS Mean vs Reading Behaviors')
ax2.set_xlim(-1, 1)

# 添加显著性标记
for i, (_, row) in enumerate(spearman_sorted.iterrows()):
    if row['significance']:
        ax2.text(row['r'] + 0.05 if row['r'] > 0 else row['r'] - 0.05, i,
                row['significance'], va='center', ha='left' if row['r'] > 0 else 'right')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'correlation_barplot.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n图表已保存: {OUTPUT_DIR / "correlation_barplot.png"}')


图表已保存: /Users/gaozhenning/Desktop/CodeData/REST_COM-main/reports/comparison/correlation_nbs_mean/correlation_barplot.png


In [17]:
# =============================================================================
# 10. 可视化 - 散点图
# =============================================================================
n_vars = len(BEHAVIOR_VARS)
n_cols = 3
n_rows = (n_vars + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten() if n_vars > 1 else [axes]

for i, var in enumerate(BEHAVIOR_VARS):
    ax = axes[i]
    
    if var not in merged_df.columns:
        ax.axis('off')
        continue
    
    x = merged_df['nbs_mean'].values.astype(float)
    y = merged_df[var].values.astype(float)
    groups = merged_df['group'].values
    
    valid_mask = ~np.isnan(x) & ~np.isnan(y)
    x_valid = x[valid_mask]
    y_valid = y[valid_mask]
    groups_valid = groups[valid_mask]
    
    # 按组别绘制散点
    for grp, color, marker in [('com', 'red', 'o'), ('td', 'blue', 's')]:
        mask = groups_valid == grp
        ax.scatter(x_valid[mask], y_valid[mask], c=color, marker=marker, 
                   label=grp.upper(), alpha=0.6, s=60)
    
    # 回归线（全体）
    if len(x_valid) >= 3:
        z = np.polyfit(x_valid, y_valid, 1)
        p = np.poly1d(z)
        x_line = np.linspace(x_valid.min(), x_valid.max(), 100)
        ax.plot(x_line, p(x_line), 'k--', linewidth=2, label='Regression')
    
    # 计算相关系数
    r, pval = stats.pearsonr(x_valid, y_valid)
    sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else ('†' if pval < 0.1 else '')))
    
    # 获取对应的Spearman相关系数
    spearman_row = spearman_df[spearman_df['behavior_var'] == var]
    rho = spearman_row['r'].values[0] if len(spearman_row) > 0 else np.nan
    rho_pval = spearman_row['p_value'].values[0] if len(spearman_row) > 0 else np.nan
    
    ax.set_xlabel('NBS Mean')
    ax.set_ylabel(var)
    
    # 显示Pearson和Spearman两种相关系数及p值
    title_text = f'{var}\nPearson: r={r:.3f}, p={pval:.4f}{sig}\nSpearman: ρ={rho:.3f}, p={rho_pval:.4f}{get_significance_marker(rho_pval)}'
    ax.set_title(title_text, fontsize=10)
    ax.legend(loc='best')

# 隐藏多余的子图
for i in range(len(BEHAVIOR_VARS), len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n散点图已保存: {OUTPUT_DIR / "scatter_plots.png"}')


散点图已保存: /Users/gaozhenning/Desktop/CodeData/REST_COM-main/reports/comparison/correlation_nbs_mean/scatter_plots.png


In [12]:
# =============================================================================
# 11. 保存结果
# =============================================================================
# 保存相关分析结果
results_df = pd.merge(
    pearson_df.rename(columns={'r': 'r_pearson'}),
    spearman_df.rename(columns={'r': 'r_spearman'}),
    on=['behavior_var', 'n'],
    suffixes=('_pearson', '_spearman')
)

# 重新整理列顺序
results_df = results_df[[
    'behavior_var', 'n',
    'r_pearson', 'p_value_pearson', 'significance_pearson',
    'r_spearman', 'p_value_spearman', 'significance_spearman'
]]

results_df.to_csv(OUTPUT_DIR / 'correlation_results.csv', index=False)

# 保存合并后的数据
merged_df.to_csv(OUTPUT_DIR / 'merged_nbs_behavior_data.csv', index=False)

print('结果已保存:')
print(f'  相关分析结果: {OUTPUT_DIR / "correlation_results.csv"}')
print(f'  合并数据: {OUTPUT_DIR / "merged_nbs_behavior_data.csv"}')
print(f'  图表: {OUTPUT_DIR / "correlation_barplot.png"}')
print(f'  散点图: {OUTPUT_DIR / "scatter_plots.png"}')

结果已保存:
  相关分析结果: /Users/gaozhenning/Desktop/CodeData/REST_COM-main/reports/comparison/correlation_nbs_mean/correlation_results.csv
  合并数据: /Users/gaozhenning/Desktop/CodeData/REST_COM-main/reports/comparison/correlation_nbs_mean/merged_nbs_behavior_data.csv
  图表: /Users/gaozhenning/Desktop/CodeData/REST_COM-main/reports/comparison/correlation_nbs_mean/correlation_barplot.png
  散点图: /Users/gaozhenning/Desktop/CodeData/REST_COM-main/reports/comparison/correlation_nbs_mean/scatter_plots.png


In [13]:
# =============================================================================
# 12. 结果汇总表格
# =============================================================================
print('\n' + '=' * 80)
print('结果汇总表')
print('=' * 80)
display_df = results_df.copy()
display_df.columns = ['行为变量', 'n', 'r(Pearson)', 'p(Pearson)', 'Sig(Pearson)', 
                       'rho(Spearman)', 'p(Spearman)', 'Sig(Spearman)']
print(display_df.to_string(index=False))


结果汇总表
   行为变量  n  r(Pearson)  p(Pearson) Sig(Pearson)  rho(Spearman)  p(Spearman) Sig(Spearman)
   150字 22   -0.244780    0.272243                   -0.182486     0.416315              
1分钟阅读平均 22   -0.330771    0.132688                   -0.390850     0.072088             †
数字RAN均值 22    0.422377    0.050202            †       0.530209     0.011140             *
物体RAN均值 22    0.427386    0.047260            *       0.477831     0.024504             *
  阅读流畅性 22   -0.381629    0.079680            †      -0.365282     0.094593             †
   音位删除 21   -0.195707    0.395220                   -0.344941     0.125678              
   部首意识 21    0.069501    0.764669                   -0.018886     0.935239              


In [14]:
# =============================================================================
# 13. 分组描述统计
# =============================================================================
print('\n' + '=' * 70)
print('NBS显著边均值的分组描述统计')
print('=' * 70)

group_stats = merged_df.groupby('group')['nbs_mean'].agg(['count', 'mean', 'std', 'min', 'max'])
print(group_stats.round(4))

# 组间差异检验
com_values = merged_df[merged_df['group'] == 'com']['nbs_mean'].dropna()
td_values = merged_df[merged_df['group'] == 'td']['nbs_mean'].dropna()

t_stat, t_pval = stats.ttest_ind(com_values, td_values)
print(f'\nCOM vs TD 组间差异检验:')
print(f'  t = {t_stat:.4f}, p = {t_pval:.4f}')

u_stat, u_pval = stats.mannwhitneyu(com_values, td_values, alternative='two-sided')
print(f'  U = {u_stat:.4f}, p = {u_pval:.4f}')


NBS显著边均值的分组描述统计
       count    mean     std     min     max
group                                       
com       14  0.3031  0.0404  0.2440  0.3737
td         8  0.2336  0.0135  0.2119  0.2560

COM vs TD 组间差异检验:
  t = 4.6738, p = 0.0001
  U = 109.0000, p = 0.0000


# 分析完成

## 主要结果
- NBS显著边数: {n_edges}
- COM组样本数: {n_com}
- TD组样本数: {n_td}

## 显著性标记说明
- ***: p < 0.001
- **: p < 0.01
- *: p < 0.05
- †: p < 0.1